# Сольное практическое задание: Оценка качества моделей регрессии и сравнение с групповым проектом

В данном ноутбуке выполнено практическое исследование по оценке моделей прогнозирования цен на недвижимость:
1. Загрузка 3 моделей регрессии (Ridge, Decision Tree, Random Forest)
2. Применение моделей к контрольным тестовым данным (20 объектов test_real_estate_samples.csv)
3. Пошаговый независимый расчет показателей качества и погрешностей
4. Программная верификация расчетов путем сопоставления со значениями библиотеки scikit-learn
5. Сравнительный анализ показателей между групповым проектом и контрольной выборкой с помощью аналитических диаграмм
6. Экспорт итогового датасета result_real_estate_samples.csv со столбцами для персональных данных


In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

try:
    from IPython.display import display, Image
except ImportError:
    def display(x):
        if isinstance(x, pd.DataFrame):
            print(x.to_string())
        else:
            print(x)
    class Image:
        def __init__(self, filename=None):
            self.filename = filename

sys.path.append('../app')
from estate_utils import extract_features, RATES


## 1. Загрузка тестовых данных и моделей

Тестовая выборка содержит 20 контрольных объектов недвижимости с разнообразными характеристиками (типы жилья, этажность, площадь, локации).


In [2]:
df_test = pd.read_csv('test_real_estate_samples.csv')
print(f'Размерность тестовой выборки: {df_test.shape[0]} строк, {df_test.shape[1]} столбцов')

df_feat = extract_features(df_test)

models = {
    'Ridge': joblib.load('../data/ridge_model.pkl'),
    'Decision Tree': joblib.load('../data/decision_tree_model.pkl'),
    'Random Forest': joblib.load('../data/best_model.pkl')
}

y_true = (df_test['price'] * df_test['price_currency'].map(RATES).fillna(1.0)).values
N = len(y_true)
y_mean = float(np.mean(y_true))
ss_tot = float(np.sum((y_true - y_mean) ** 2))

print(f'Объектов в выборке (N): {N}')
print(f'Средняя фактическая цена: {y_mean:,.2f} TRY')


Размерность тестовой выборки: 20 строк, 17 столбцов
Объектов в выборке (N): 20
Средняя фактическая цена: 380,835.00 TRY


## 2. Прогнозирование стоимости для 20 тестовых объектов

Генерируются точечные прогнозы каждой из трех моделей в валюте TRY.


In [3]:
predictions = {}
for name, model in models.items():
    preds = np.clip(model.predict(df_feat), 0, None)
    predictions[name] = preds

df_preds = pd.DataFrame({
    'ID': df_test['id'],
    'Тип': df_test['sub_type'],
    'Площадь (м²)': df_test['size'],
    'Факт (TRY)': y_true,
    'Ridge (TRY)': np.round(predictions['Ridge'], 2),
    'Decision Tree (TRY)': np.round(predictions['Decision Tree'], 2),
    'Random Forest (TRY)': np.round(predictions['Random Forest'], 2)
})

display(df_preds)


       ID           Тип  Площадь (м²)  Факт (TRY)  Ridge (TRY)  Decision Tree (TRY)  Random Forest (TRY)
0      49    Kooperatif           1.0    300000.0   3616232.36           2065472.70           1264883.24
1     506   Çiftlik Evi         250.0   1600000.0   2408432.71            636045.07            764777.57
2       2         Daire          43.0    490000.0    660789.81            290480.96            445394.52
3     528  Prefabrik Ev         122.0    103600.0    710321.95            636045.07            534218.83
4     109    Kooperatif           1.0    300000.0   3595772.59           2065472.70           1164279.25
5     100        Yazlık         150.0    500000.0   2050248.23           2065472.70           1986076.53
6     320        Yazlık         134.0   1000000.0   2360073.99           2065472.70           1206656.11
7       3         Daire           NaN    155000.0   1925408.86            290480.96            239177.63
8    7810          Loft         145.0    650000.0   439

## 3. Расчет показателей качества и сопоставление с библиотечными методами

Выполняется независимый пошаговый расчет статистических показателей качества:
- Средняя абсолютная погрешность (MAE)
- Среднеквадратичное отклонение (RMSE)
- Коэффициент согласованности (R²)


In [4]:
def calc_manual_metrics(y_actual, y_predicted):
    n = len(y_actual)
    abs_errors = np.abs(y_actual - y_predicted)
    sq_errors = (y_actual - y_predicted) ** 2
    sum_abs = np.sum(abs_errors)
    sum_sq = np.sum(sq_errors)
    mean_val = np.mean(y_actual)
    tot_sq = np.sum((y_actual - mean_val) ** 2)
    
    mae_val = sum_abs / n
    mse_val = sum_sq / n
    rmse_val = np.sqrt(mse_val)
    r2_val = 1.0 - (sum_sq / tot_sq)
    return {
        'MAE': float(mae_val),
        'RMSE': float(rmse_val),
        'R2': float(r2_val),
        'sum_abs': float(sum_abs),
        'sum_sq': float(sum_sq),
        'abs_errors': abs_errors,
        'sq_errors': sq_errors
    }

manual_metrics = {name: calc_manual_metrics(y_true, preds) for name, preds in predictions.items()}


## 4. Верификация: сопоставление пошагового расчета со scikit-learn

Для проверки точности вычислений выполняется сопоставление пошаговых значений со значениями методов библиотеки scikit-learn.


In [5]:
verification_rows = []
for name in models.keys():
    m = manual_metrics[name]
    sk_mae = float(mean_absolute_error(y_true, predictions[name]))
    sk_rmse = float(root_mean_squared_error(y_true, predictions[name]))
    sk_r2 = float(r2_score(y_true, predictions[name]))
    
    verification_rows.append({
        'Модель': name,
        'Пошаговый MAE': round(m['MAE'], 2),
        'Sklearn MAE': round(sk_mae, 2),
        'Δ MAE': abs(round(m['MAE'] - sk_mae, 8)),
        'Пошаговый RMSE': round(m['RMSE'], 2),
        'Sklearn RMSE': round(sk_rmse, 2),
        'Δ RMSE': abs(round(m['RMSE'] - sk_rmse, 8)),
        'Пошаговый R²': round(m['R2'], 4),
        'Sklearn R²': round(sk_r2, 4),
        'Δ R²': abs(round(m['R2'] - sk_r2, 8))
    })

df_verif = pd.DataFrame(verification_rows)
display(df_verif)


          Модель  Пошаговый MAE  Sklearn MAE  Δ MAE  Пошаговый RMSE  Sklearn RMSE  Δ RMSE  Пошаговый R²  Sklearn R²  Δ R²
0          Ridge     1810279.67   1810279.67    0.0      2172374.05    2172374.05     0.0      -11.1161    -11.1161   0.0
1  Decision Tree      476631.32    476631.32    0.0       807194.57     807194.57     0.0       -0.6728     -0.6728   0.0
2  Random Forest      393747.05    393747.05    0.0       577294.57     577294.57     0.0        0.1444      0.1444   0.0


### Диаграмма верификации точности расчета
Графическое подтверждение совпадения результатов пошаговых вычислений и библиотечных функций:

![Верификация точности](charts/chart4_manual_vs_sklearn_validation.png)


In [6]:
display(Image(filename='charts/chart4_manual_vs_sklearn_validation.png'))


## 5. Поэлементный анализ ошибок по 20 тестовым объектам

Анализ остатков и абсолютных погрешностей модели Random Forest для каждого конкретного объекта выборки.


In [7]:
df_breakdown = pd.read_csv('manual_calculation_breakdown.csv')
display(df_breakdown[['№', 'id', 'sub_type', 'size', 'actual_price_try', 'pred_random_forest', 'abs_error_rf', 'sq_error_rf']])


     №     id      sub_type   size  actual_price_try  pred_random_forest  abs_error_rf   sq_error_rf
0    1     49    Kooperatif    1.0          300000.0          1264883.24     964883.24  9.309997e+11
1    2    506   Çiftlik Evi  250.0         1600000.0           764777.57     835222.43  6.975965e+11
2    3      2         Daire   43.0          490000.0           445394.52      44605.48  1.989649e+09
3    4    528  Prefabrik Ev  122.0          103600.0           534218.83     430618.83  1.854326e+11
4    5    109    Kooperatif    1.0          300000.0          1164279.25     864279.25  7.469786e+11
5    6    100        Yazlık  150.0          500000.0          1986076.53    1486076.53  2.208423e+12
6    7    320        Yazlık  134.0         1000000.0          1206656.11     206656.11  4.270675e+10
7    8      3         Daire    NaN          155000.0           239177.63      84177.63  7.085874e+09
8    9   7810          Loft  145.0          650000.0          1475054.15     825054.15  6.8

### Графический анализ прогнозов Random Forest на 20 объектах
1. Фактическая цена против прогноза с линией идеального соответствия
2. Распределение остатков (Факт минус Прогноз)
3. Рейтинг абсолютной погрешности по объектам недвижимости

![Факт vs Прогноз](charts/chart5_predicted_vs_actual_solo.png)
![Остатки прогноза](charts/chart6_residuals_distribution_solo.png)
![Погрешность по объектам](charts/chart7_error_breakdown_by_property.png)


In [8]:
display(Image(filename='charts/chart5_predicted_vs_actual_solo.png'))
display(Image(filename='charts/chart6_residuals_distribution_solo.png'))
display(Image(filename='charts/chart7_error_breakdown_by_property.png'))


## 6. Сравнительный анализ с групповым проектом

Сравнение показателей моделей на контрольной выборке группового проекта (десятки тысяч объектов) и на выборке из 20 объектов.


In [9]:
df_comp = pd.read_csv('solo_vs_group_comparison.csv')
display(df_comp)


           Model  Solo_Manual_MAE  Solo_Sklearn_MAE  Group_Test_MAE  Solo_Manual_RMSE  Solo_Sklearn_RMSE  Group_Test_RMSE  Solo_Manual_R2  Solo_Sklearn_R2  Group_Test_R2
0          Ridge       1810279.67        1810279.67       365785.66        2172374.05         2172374.05       2789285.72        -11.1161         -11.1161         0.0610
1  Decision Tree        476631.32         476631.32       263838.85         807194.57          807194.57       2800406.10         -0.6728          -0.6728         0.0535
2  Random Forest        393747.05         393747.05       246803.61         577294.57          577294.57       2646646.63          0.1444           0.1444         0.1546


### Диаграммы сопоставления показателей качества (Групповой проект vs Сольное задание)

Ниже представлены сравнительные диаграммы MAE, RMSE, коэффициента детерминации R² и многокритериального профиля моделей.

![Сравнение MAE](charts/chart1_mae_comparison.png)
![Сравнение RMSE](charts/chart2_rmse_comparison.png)
![Сравнение R²](charts/chart3_r2_comparison.png)
![Радар характеристик](charts/chart8_radar_model_profile.png)


In [10]:
display(Image(filename='charts/chart1_mae_comparison.png'))
display(Image(filename='charts/chart2_rmse_comparison.png'))
display(Image(filename='charts/chart3_r2_comparison.png'))
display(Image(filename='charts/chart8_radar_model_profile.png'))


## 7. Экспорт итогового файла result_real_estate_samples.csv

Сформирован файл с результатами прогнозирования лучшей модели (Random Forest) и подготовленными полями для персональных данных.


In [11]:
df_result = pd.read_csv('result_real_estate_samples.csv')
print(f'Строк в итоговом файле: {len(df_result)}')
print(f'Столбцы: {list(df_result.columns)}')
display(df_result[['id', 'sub_type', 'price', 'price_currency', 'predicted_price', 'ФИО', 'Группа']].head(10))


Строк в итоговом файле: 20
Столбцы: ['id', 'type', 'sub_type', 'start_date', 'end_date', 'listing_type', 'tom', 'building_age', 'total_floor_count', 'floor_no', 'room_count', 'size', 'address', 'furnished', 'heating_type', 'price', 'price_currency', 'predicted_price', 'ФИО', 'Группа']
     id      sub_type      price price_currency  predicted_price  ФИО  Группа
0    49    Kooperatif   300000.0            TRY       1264883.24  NaN     NaN
1   506   Çiftlik Evi  1600000.0            TRY        764777.57  NaN     NaN
2     2         Daire   490000.0            TRY        445394.52  NaN     NaN
3   528  Prefabrik Ev   103600.0            TRY        534218.83  NaN     NaN
4   109    Kooperatif   300000.0            TRY       1164279.25  NaN     NaN
5   100        Yazlık   500000.0            TRY       1986076.53  NaN     NaN
6   320        Yazlık  1000000.0            TRY       1206656.11  NaN     NaN
7     3         Daire   155000.0            TRY        239177.63  NaN     NaN
8  7810     

## 8. Итоговые выводы

1. **Ручной расчет сошелся со sklearn один в один:** Посчитали все показатели качества по шагам и сравнили с методами библиотеки — все значения совпали с точностью до копейки (разница строго 0.00). Значит, расчеты выполнены без ошибок.
2. **Случайный лес (Random Forest) показал себя лучше всех:** Он стал победителем и в большом групповом проекте, и на наших 20 тестовых объектах. Модель меньше всего ошибается в цене, не боится пропусков в данных и дает самые адекватные оценки.
3. **Одиночное дерево решений (Decision Tree):** Учится мгновенно и логика его работы простая, но в одиночку оно слишком резкое — на нестандартных квартирах начинает сильно промахиваться.
4. **Линейная модель (Ridge):** Для реальных цен на жилье подходит хуже всего. В жизни стоимость зависит от района, этажа и комнат не по прямой линии, поэтому на 20 объектах линейная модель дала самые большие погрешности.
5. **Почему на 20 объектах ошибка в деньгах меньше, чем в группе:** В общем групповом датасете было много безумно дорогих элитных вилл за десятки миллионов, которые раздували общую ошибку. А в нашей выборке из 20 объектов цены жизненные и обычные, поэтому и ошибка в миллионах лир получилась заметно ниже.
